# 12. FastSLAM 개념 구현

FastSLAM은 SLAM posterior를 Rao-Blackwellization으로 분해한다.

$$p(x_{1:t},m\mid z_{1:t},u_{1:t}) = p(x_{1:t}\mid z,u)\prod_j p(m_j\mid x_{1:t},z)$$

즉 trajectory는 particle로, 각 landmark는 particle 내부의 작은 EKF로 추정한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Particle trajectory + landmark EKF

간단화를 위해 landmark id는 알고 있다고 가정한다. 각 particle은 자기만의 landmark 평균/공분산을 가진다.

In [ ]:
np.random.seed(24)
def wrap(a): return np.arctan2(np.sin(a),np.cos(a))
landmarks=np.array([[2,1],[5,1.2],[4,4]])
M=120; N=len(landmarks)
particles=np.zeros((M,3)); particles[:,0]=np.random.randn(M)*0.15; particles[:,1]=np.random.randn(M)*0.15; particles[:,2]=np.random.randn(M)*0.08
lm_mu=np.zeros((M,N,2)); lm_P=np.tile(np.eye(2)*1000,(M,N,1,1)); seen=np.zeros((M,N),dtype=bool); weights=np.ones(M)/M
true=np.array([0.0,0.0,0.0]); Q=np.diag([0.15**2,np.deg2rad(5)**2])
controls=[(0.45,0.16)]*18+[(0.45,-0.1)]*18

def motion(x,u,noise=True):
    v,w=u; dt=0.2
    if noise: v+=np.random.randn()*0.04; w+=np.random.randn()*0.03
    return np.array([x[0]+v*dt*np.cos(x[2]), x[1]+v*dt*np.sin(x[2]), wrap(x[2]+w*dt)])
def h(x,m):
    dx=m[0]-x[0]; dy=m[1]-x[1]
    return np.array([np.hypot(dx,dy), wrap(np.arctan2(dy,dx)-x[2])])
def H_lm(x,m):
    dx=m[0]-x[0]; dy=m[1]-x[1]; q=dx*dx+dy*dy; r=np.sqrt(q)
    return np.array([[dx/r,dy/r],[-dy/q,dx/q]])
def init_lm(x,z):
    r,b=z; ang=x[2]+b
    return np.array([x[0]+r*np.cos(ang), x[1]+r*np.sin(ang)])
def resample(w):
    idx=np.random.choice(len(w),size=len(w),p=w)
    return idx

true_hist=[]; mean_hist=[]
for u in controls:
    true=motion(true,u,noise=False)
    zs=[h(true,m)+np.random.multivariate_normal([0,0],Q) for m in landmarks]
    for i in range(M):
        particles[i]=motion(particles[i],u,noise=True)
        wi=1.0
        for j,z in enumerate(zs):
            z[1]=wrap(z[1])
            if not seen[i,j]:
                lm_mu[i,j]=init_lm(particles[i],z)
                H=H_lm(particles[i],lm_mu[i,j])
                lm_P[i,j]=np.linalg.inv(H)@Q@np.linalg.inv(H).T
                seen[i,j]=True
            else:
                zhat=h(particles[i],lm_mu[i,j]); H=H_lm(particles[i],lm_mu[i,j])
                S=H@lm_P[i,j]@H.T+Q; innov=z-zhat; innov[1]=wrap(innov[1])
                wi*=float(np.exp(-0.5*innov@np.linalg.inv(S)@innov)/np.sqrt(np.linalg.det(2*np.pi*S)))
                K=lm_P[i,j]@H.T@np.linalg.inv(S)
                lm_mu[i,j]=lm_mu[i,j]+K@innov
                lm_P[i,j]=(np.eye(2)-K@H)@lm_P[i,j]
        weights[i]=wi+1e-300
    weights/=weights.sum()
    idx=resample(weights); particles=particles[idx]; lm_mu=lm_mu[idx]; lm_P=lm_P[idx]; seen=seen[idx]; weights=np.ones(M)/M
    true_hist.append(true.copy()); mean_hist.append(particles.mean(axis=0))
true_hist=np.array(true_hist); mean_hist=np.array(mean_hist)
lm_mean=lm_mu.mean(axis=0)

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(true_hist[:,0],true_hist[:,1],'k-',lw=2,label='true')
axes[0].plot(mean_hist[:,0],mean_hist[:,1],color='#E85D24',lw=2,label='particle mean')
axes[0].scatter(particles[:,0],particles[:,1],s=12,color='#534AB7',alpha=0.35,label='particles')
axes[0].scatter(landmarks[:,0],landmarks[:,1],marker='*',s=160,color='#1D9E75',label='true landmarks')
axes[0].scatter(lm_mean[:,0],lm_mean[:,1],s=90,color='#E85D24',label='mean lm')
axes[0].axis('equal'); axes[0].grid(alpha=0.25); axes[0].legend(); axes[0].set_title('FastSLAM simplified')
axes[1].bar(np.arange(N),np.linalg.norm(lm_mean-landmarks,axis=1),color='#534AB7')
axes[1].set_title('landmark mean error'); axes[1].set_xlabel('landmark id'); axes[1].grid(axis='y',alpha=0.25)
plt.tight_layout(); plt.savefig('assets/12_fastslam_simplified.png',dpi=150,bbox_inches='tight'); plt.show()
print('final pose mean:', np.round(mean_hist[-1],3))
print('final true pose:', np.round(true_hist[-1],3))

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Rao-Blackwellization | trajectory와 map posterior 분해 | Ch.13 FastSLAM |
| Particle path | robot trajectory 가설 | MCL의 확장 |
| Landmark EKF | particle별 landmark 추정 | mapping을 조건부 독립화 |